In [ ]:
# NYC taxi: neural net, trained in its own kernel
# Run NYC-TAXI-FINAL_1.ipynb first: its train/test split cell saves model_data/*.parquet.
# A fresh kernel holds only the split and this small network, so it can't run out of
# memory the way the big notebook did, and XGBoost's OpenMP library is never loaded.
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # harmless here; protects against libomp clashes

import gc
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATA_DIR = Path("model_data")
MODEL_DIR = Path("saved_models")
MODEL_DIR.mkdir(exist_ok=True)

CAT_COLS = ["VendorID", "PULocationID", "DOLocationID"]
NUM_COLS = ["pickup_hour", "pickup_month", "pickup_day", "trip_distance", "duration_min"]

X_train = pd.read_parquet(DATA_DIR / "X_train.parquet")
X_test = pd.read_parquet(DATA_DIR / "X_test.parquet")
y_train = pd.read_parquet(DATA_DIR / "y_train.parquet").squeeze("columns")
y_test = pd.read_parquet(DATA_DIR / "y_test.parquet").squeeze("columns")
print(f"Loaded {len(X_train):,} train / {len(X_test):,} test rows")

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def subsample(X, y, n, seed=42):
    """Same sampler as the main notebook, so the NN sees the same rows as before."""
    if n >= len(X):
        return X, y
    idx = np.sort(np.random.default_rng(seed).choice(len(X), size=n, replace=False))
    return X.iloc[idx], y.iloc[idx]

In [ ]:
# %% 5f. Neural net with embeddings for zones/vendor
# Crash-proofing for macOS (changes from the original version of this cell):
#   1. KMP_DUPLICATE_LIB_OK: XGBoost (loaded in an earlier cell) and PyTorch each
#      ship their own OpenMP library; loading both makes macOS abort Python with
#      "OMP: Error #15". This must be set BEFORE torch is imported.
#   2. Runs on CPU by default. Apple's MPS backend can hard-crash Python instead of
#      raising an error. This network is small, so CPU takes only a few minutes.
#   3. Embedding sizes come from the data. An ID larger than the table (e.g. a new
#      VendorID) is a clean error on CPU but can crash the process on MPS.
#   4. Caps torch's thread count so it doesn't fight sklearn/XGBoost threads.
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
torch.set_num_threads(4)

USE_ACCELERATOR = False   # set True to try CUDA/MPS once the CPU run works
device = ("cuda" if USE_ACCELERATOR and torch.cuda.is_available()
          else "mps" if USE_ACCELERATOR and torch.backends.mps.is_available()
          else "cpu")
print("Training on:", device)

NN_TRAIN_ROWS = 2_000_000   # subsample for speed
EPOCHS, BATCH = 15, 4096

# Embedding table sizes: largest ID seen anywhere (train or test) + 1
CARD = {c: int(max(X_train[c].max(), X_test[c].max())) + 1 for c in CAT_COLS}
print("Embedding table sizes:", CARD)

num_mean, num_std = X_train[NUM_COLS].mean(), X_train[NUM_COLS].std()
y_mean, y_std = float(y_train.mean()), float(y_train.std())

def to_tensors(X):
    cat = torch.tensor(X[CAT_COLS].to_numpy(dtype="int64"))
    num = torch.tensor(((X[NUM_COLS] - num_mean) / num_std).to_numpy(dtype="float32"))
    return cat, num

class FareNet(nn.Module):
    def __init__(self, n_num, card):
        super().__init__()
        self.veh = nn.Embedding(card["VendorID"], 2)
        self.pu = nn.Embedding(card["PULocationID"], 16)
        self.do = nn.Embedding(card["DOLocationID"], 16)
        self.mlp = nn.Sequential(
            nn.Linear(2 + 16 + 16 + n_num, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, cat, num):
        z = torch.cat([self.veh(cat[:, 0]), self.pu(cat[:, 1]), self.do(cat[:, 2]), num], dim=1)
        return self.mlp(z).squeeze(-1)

nn_X, nn_y = subsample(X_train, y_train, NN_TRAIN_ROWS)
nn_Xtr, nn_Xval, nn_ytr, nn_yval = train_test_split(nn_X, nn_y, test_size=0.05, random_state=42)

def make_loader(X, y, shuffle):
    cat, num = to_tensors(X)
    tgt = torch.tensor(((y - y_mean) / y_std).to_numpy(dtype="float32"))
    return DataLoader(TensorDataset(cat, num, tgt), batch_size=BATCH, shuffle=shuffle)

train_loader = make_loader(nn_Xtr, nn_ytr, True)
val_loader = make_loader(nn_Xval, nn_yval, False)
del nn_X, nn_y, nn_Xtr, nn_Xval, nn_ytr, nn_yval   # loaders hold their own copies

net = FareNet(len(NUM_COLS), CARD).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=1)
loss_fn = nn.MSELoss()
best_rmse, best_state = float("inf"), None

for epoch in range(EPOCHS):
    net.train()
    for cat, num, tgt in train_loader:
        cat, num, tgt = cat.to(device), num.to(device), tgt.to(device)
        optimizer.zero_grad()
        loss_fn(net(cat, num), tgt).backward()
        optimizer.step()

    net.eval()
    sq_err, n = 0.0, 0
    with torch.no_grad():
        for cat, num, tgt in val_loader:
            pred = net(cat.to(device), num.to(device)).cpu()
            sq_err += ((pred - tgt) ** 2).sum().item()
            n += len(tgt)
    val_rmse = (sq_err / n) ** 0.5 * y_std      # back to dollars
    scheduler.step(val_rmse)
    if val_rmse < best_rmse:
        best_rmse = val_rmse
        best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
    print(f"Epoch {epoch + 1:2d}  val RMSE ${val_rmse:.4f}")

net.load_state_dict(best_state)
net.eval()

def nn_predict(X, chunk=262_144):
    out = []
    with torch.no_grad():
        for i in range(0, len(X), chunk):
            cat, num = to_tensors(X.iloc[i:i + chunk])
            out.append(net(cat.to(device), num.to(device)).cpu().numpy())
    return np.concatenate(out) * y_std + y_mean

In [ ]:
# Test-set predictions -> saved for the main notebook's results table and plots
nn_test_pred = nn_predict(X_test).astype("float32")
print("NeuralNet", {
    "RMSE": round(float(rmse(y_test, nn_test_pred)), 4),
    "MAE": round(float(mean_absolute_error(y_test, nn_test_pred)), 4),
    "R2": round(float(r2_score(y_test, nn_test_pred)), 4),
})
np.save(MODEL_DIR / "nn_test_preds.npy", nn_test_pred)
np.save(MODEL_DIR / "nn_test_index.npy", X_test.index.to_numpy())   # lets the main notebook check alignment
torch.save(net.state_dict(), MODEL_DIR / "nn_state.pt")
print(f"Saved predictions and weights to {MODEL_DIR}/")

In [ ]:
def permutation_importance(predict, X, y, n_repeats=3, seed=42):
    rng = np.random.default_rng(seed)
    base_pred = predict(X)
    base_rmse, base_mae = rmse(y, base_pred), mean_absolute_error(y, base_pred)
    out = {"RMSE": {}, "MAE": {}}
    for col in X.columns:
        r_scores, m_scores = [], []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[col] = rng.permutation(Xp[col].to_numpy())
            p = predict(Xp)
            r_scores.append(rmse(y, p))
            m_scores.append(mean_absolute_error(y, p))
        out["RMSE"][col] = np.mean(r_scores) - base_rmse
        out["MAE"][col] = np.mean(m_scores) - base_mae
    return pd.DataFrame(out)

# Same 200k-row sample and same shuffling seed as the main notebook's importance cell
X_imp = X_test.sample(200_000, random_state=0)
y_imp = y_test.loc[X_imp.index]
nn_imp = permutation_importance(nn_predict, X_imp, y_imp)
nn_imp.to_parquet(MODEL_DIR / "nn_importance.parquet")
print(nn_imp.sort_values("RMSE", ascending=False).round(4))